In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as plt
import seaborn as sns

import json
import geopandas as gpd

import os

In [208]:
mapping_kecamatan = {
    "ASEMROWO": "ASEM ROWO",
    "DUKUHPAKIS": "DUKUH PAKIS",
    "GUNUNGANYAR": "GUNUNG ANYAR",
    "KARANGPILANG": "KARANG PILANG",
    "LAKARSANTRI": "LAKARSANTRI",
    "MULYOREJO": "MULYOREJO",
    "SAMBIKEREP": "SAMBIKEREP",
    "SUKOLILO": "SUKOLILO",
    "SUKOMANUNGGAL": "SUKOMANUNGGAL",
    "TENGGILISMEJOYO": "TENGGILIS MEJOYO",
    "WIYUNG": "WIYUNG",

    # Nama yang memang sudah sama
    "BENOWO": "BENOWO",
    "PAKAL": "PAKAL",
    "BUBUTAN": "BUBUTAN",
    "BULAK": "BULAK",
    "GAYUNGAN": "GAYUNGAN",
    "GENTENG": "GENTENG",
    "GUBENG": "GUBENG",
    "JAMBANGAN": "JAMBANGAN",
    "KENJERAN": "KENJERAN",
    "KREMBANGAN": "KREMBANGAN",
    "PABEAN CANTIAN": "PABEAN CANTIAN",
    "RUNGKUT": "RUNGKUT",
    "SAWAHAN": "SAWAHAN",
    "SEMAMPIR": "SEMAMPIR",
    "SIMOKERTO": "SIMOKERTO",
    "TAMBAKSARI": "TAMBAKSARI",
    "TANDES": "TANDES",
    "TEGALSARI": "TEGALSARI",
    "WONOCOLO": "WONOCOLO",
    "WONOKROMO": "WONOKROMO"
}

def normalisasi_kecamatan(df, kolom="Kecamatan"):
    df[kolom] = (
        df[kolom]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)   # rapikan spasi
    )

    mapping = {
        "ASEMROWO": "ASEM ROWO",
        "DUKUHPAKIS": "DUKUH PAKIS",
        "GUNUNGANYAR": "GUNUNG ANYAR",
        "KARANGPILANG": "KARANG PILANG",
        "TENGGILISMEJOYO": "TENGGILIS MEJOYO",
    }

    df[kolom] = df[kolom].replace(mapping)
    return df

In [264]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    lat1, lon1 = radians(lat1), radians(lon1)
    lat2, lon2 = radians(lat2), radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

def idw(df, target_kecamatan, value_col, power=2):

    # lokasi target
    target = df[df["Kecamatan"] == target_kecamatan].iloc[0]

    # hanya data yang diketahui
    known = df[df[value_col].notna()].copy()

    # hitung jarak
    known["distance"] = known.apply(
        lambda row: haversine(
            target["lat"],
            target["long"],
            row["lat"],
            row["long"]
        ),
        axis=1
    )

    # hindari pembagian nol
    known = known[known["distance"] > 0]

    # bobot IDW
    known["weight"] = 1 / (known["distance"] ** power)

    # prediksi
    pred = (
        (known["weight"] * known[value_col]).sum()
        / known["weight"].sum()
    )

    return pred
koordinat = pd.read_excel("D:\KULIAH\Project\womanguard-index-surabaya\Koordinat.xlsx")
koordinat.head()

,No,Kecamatan,lat,long
0,1,Karang Pilang,-7.341345,112.695990
1,2,Wonocolo,-7.319820,112.742027
2,3,Rungkut,-7.319019,112.804593
3,4,Wonokromo,-7.303957,112.736011
4,5,Tegalsari,-7.279848,112.736069


# X1

In [290]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x1_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [291]:
jumlah_penduduk = pd.read_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin\jumlah_penduduk_surabaya_per_kecamatan_2023_2025.csv")

In [292]:
#Imputasi Inverse Distance Weighting (IDW)
data = x1_2024.merge(koordinat, on="Kecamatan")
prediksi = idw(
    data,
    target_kecamatan="Tandes",
    value_col="Jumlah (Jiwa)",
    power=2
)

print(prediksi)

2165.438196269988


In [293]:
x1_2024.loc[
    x1_2024["Kecamatan"]=="Tandes",
    "Jumlah (Jiwa)"
] = round(prediksi)

In [321]:
gabungan = []

# Normalisasi data jumlah penduduk cukup sekali
jumlah_penduduk = normalisasi_kecamatan(jumlah_penduduk)

for tahun in range(2023, 2026):
    df = globals()[f"x1_{tahun}"].copy()

    # Format data x1 berbeda
    if tahun == 2023:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 0],
            "Tahun": tahun,
            "Keluarga Miskin": pd.to_numeric(df.iloc[:, 1], errors="coerce")
        })
    else:
        temp = pd.DataFrame({
            "Kecamatan": df["Kecamatan"],
            "Tahun": tahun,
            "Keluarga Miskin": pd.to_numeric(df["Jumlah (Jiwa)"], errors="coerce")
        })

    # Normalisasi nama kecamatan
    temp = normalisasi_kecamatan(temp)

    # Hapus baris total kota
    temp = temp[
        ~temp["Kecamatan"].isin(["JUMLAH", "KOTA SURABAYA"])
    ].reset_index(drop=True)

    # Merge berdasarkan Kecamatan dan Tahun
    temp = temp.merge(
        jumlah_penduduk,
        on=["Kecamatan", "Tahun"],
        how="left"
    )

    # Hitung rasio
    temp["Rasio Kemiskinan"] = (
        temp["Keluarga Miskin"] /
        temp["Jumlah Penduduk"]
    ) * 100

    gabungan.append(temp)

# Gabungkan seluruh tahun
x1_final = pd.concat(gabungan, ignore_index=True)

# Export
x1_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin\X1_Rasio_Keluarga_Miskin.csv",
    index=False,
    encoding="utf-8-sig"
)

x1_final.head()

,Kecamatan,Tahun,Keluarga Miskin,Jumlah Penduduk,Rasio Kemiskinan
0,TAMBAKSARI,2023,19654,226995,8.658340
1,WONOKROMO,2023,11973,154995,7.724765
2,SUKOMANUNGGAL,2023,3410,104786,3.254252
3,SEMAMPIR,2023,15171,182371,8.318757
4,GUBENG,2023,9613,133804,7.184389


# X2

In [317]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X2 Persentase Kepala Keluarga Perempuan"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x2_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [310]:
for tahun in range(2023, 2026):
    print(f"===== {tahun} =====")
    print(globals()[f"x2_{tahun}"].columns.tolist())

===== 2023 =====
['No', 'Kecamatan', 'Laki-laki', 'Perempuan', 'Jumlah']
===== 2024 =====
['No', 'Kecamatan', 'Laki-laki', 'Perempuan', 'Jumlah']
===== 2025 =====
['Kecamtan', 'Kepala KeluargaPerempuan', 'Kepala keluarga laki laki']


In [318]:
gabungan = []

for tahun in range(2023, 2026):
    df = globals()[f"x2_{tahun}"].copy()

    if tahun != 2025:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 1],
            "Tahun": tahun,
            "Rasio Kepala Keluarga Perempuan": df.iloc[:, 3]/(df.iloc[:,2] + df.iloc[:,3])*100
        })
    else:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 0],
            "Tahun": tahun,
            "Rasio Kepala Keluarga Perempuan": df.iloc[:, 1]/(df.iloc[:,1] + df.iloc[:,2])*100
        })

    # Tambahkan ke list untuk semua tahun
    gabungan.append(temp)

# Gabungkan semua tahun
x2_final = pd.concat(gabungan, ignore_index=True)

# Export ke CSV
x2_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X2 Persentase Kepala Keluarga Perempuan\X2_Rasio Kepala Keluarga Perempuan.csv",
    index=False,
    encoding="utf-8-sig"
)
x2_final.head()

,Kecamatan,Tahun,Rasio Kepala Keluarga Perempuan
0,Karang Pilang,2023,22.031423
1,Wonocolo,2023,23.825202
2,Rungkut,2023,21.401099
3,Wonokromo,2023,27.068295
4,Tegalsari,2023,28.310238


# X3

In [429]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x3_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [430]:
gabungan = []

for tahun in range(2023, 2026):

    df = globals()[f"x3_{tahun}"].copy()

    # ==========================
    # Gabungkan header
    # ==========================
    header1 = df.iloc[3].fillna("").astype(str)
    header2 = df.iloc[4].fillna("").astype(str)

    kolom = []

    for h1, h2 in zip(header1, header2):

        h1 = h1.strip()
        h2 = h2.strip()

        if h1 == "":
            kolom.append(h2)
        elif h2 == "":
            kolom.append(h1)
        else:
            kolom.append(f"{h1}_{h2}")

    df.columns = kolom

    # Hapus header
    df = df.iloc[5:].reset_index(drop=True)

    # ==========================
    # Perbaiki kolom PR
    # ==========================
    kolom_baru = []
    nama = None

    for c in df.columns:

        if c.endswith("_LK"):
            nama = c[:-3]
            kolom_baru.append(c)

        elif c == "PR":
            kolom_baru.append(f"{nama}_PR")

        else:
            kolom_baru.append(c)

    df.columns = (
        pd.Index(kolom_baru)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # ==========================
    # Ambil kecamatan saja
    # ==========================
    df = df[df.iloc[:, 0] == "KECAMATAN"].reset_index(drop=True)

    # Numerik
    for c in df.columns[2:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Hilangkan fragmentation
    df = df.copy()

    # ==========================
    # Tidak bekerja
    # ==========================
    tidak_bekerja = [
        "BELUM_TIDAK_BEKERJA_PR",
        "MENGURUS_RUMAH_TANGGA_PR",
        "PELAJAR_MAHASISWA_PR",
        "PENSIUNAN_PR"
    ]

    # ==========================
    # Seluruh perempuan bekerja
    # ==========================
    bekerja_cols = [
        c for c in df.columns
        if c.endswith("_PR")
        and c not in tidak_bekerja
    ]

    # ==========================
    # Pekerjaan sektor informal
    # ==========================
    informal = [
        "PERDAGANGAN",
        "PETANI_PEKEBUN",
        "PETERNAK",
        "NELAYAN_PERIKANAN",
        "BURUH_HARIAN_LEPAS",
        "BURUH_TANI_PERKEBUNAN",
        "BURUH_NELAYAN_PERIKANAN",
        "BURUH_PETERNAKAN",
        "PEMBANTU_RUMAH_TANGGA",
        "TUKANG_CUKUR",
        "TUKANG_LISTRIK",
        "TUKANG_BATU",
        "TUKANG_KAYU",
        "TUKANG_SOL_SEPATU",
        "TUKANG_LAS_PANDAI_BESI",
        "TUKANG_JAHIT",
        "TUKANG_GIGI",
        "PENATA_RIAS",
        "PENATA_BUSANA",
        "PENATA_RAMBUT",
        "MEKANIK",
        "SENIMAN",
        "TABIB",
        "PARAJI",
        "PERANCANG_BUSANA",
        "PENTERJEMAH",
        "JURU_MASAK",
        "PROMOTOR_ACARA",
        "SOPIR",
        "PARANORMAL",
        "PEDAGANG",
        "WIRASWASTA"
    ]

    informal_cols = [
        c for c in bekerja_cols
        if any(job in c for job in informal)
    ]

    # ==========================
    # Hitung indikator
    # ==========================
    hasil = pd.DataFrame({
        "Perempuan Bekerja": df[bekerja_cols].sum(axis=1),
        "Perempuan Sektor Informal": df[informal_cols].sum(axis=1)
    })

    hasil["Persentase Perempuan Sektor Informal"] = np.where(
        hasil["Perempuan Bekerja"] == 0,
        np.nan,
        hasil["Perempuan Sektor Informal"] /
        hasil["Perempuan Bekerja"] * 100
    )

    temp = pd.concat([
        df["NAMA_KELURAHAN"].rename("Kecamatan"),
        pd.Series(tahun, index=df.index, name="Tahun"),
        hasil
    ], axis=1)

    gabungan.append(temp)

# Gabungkan seluruh tahun
x3_final = pd.concat(gabungan, ignore_index=True)

# Export
x3_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X3 Persentase Perempuan Bekerja di Sektor Informal\X3_Persentase_Perempuan_Bekerja_di_Sektor_Informal.csv",
    index=False,
    encoding="utf-8-sig"
)

x3_final.head()

,Kecamatan,Tahun,Perempuan Bekerja,Perempuan Sektor Informal,Persentase Perempuan Sektor Informal
0,KARANG PILANG,2023,2037,389,19.096711
1,WONOCOLO,2023,2119,512,24.162341
2,RUNGKUT,2023,3497,712,20.360309
3,WONOKROMO,2023,5221,1105,21.164528
4,TEGALSARI,2023,3236,578,17.861557


# X4

In [363]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x4_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [364]:
gabungan = []

for tahun in range(2023, 2026):

    df = globals()[f"x4_{tahun}"].copy()

    # ==========================
    # Gabungkan header
    # ==========================

    header1 = df.iloc[3].fillna("").astype(str)
    header2 = df.iloc[4].fillna("").astype(str)

    kolom = []

    for h1, h2 in zip(header1, header2):

        h1 = h1.strip()
        h2 = h2.strip()

        if h1 == "":
            kolom.append(h2)

        elif h2 == "":
            kolom.append(h1)

        else:
            kolom.append(f"{h1}_{h2}")

    df.columns = kolom

    # hapus header
    df = df.iloc[5:].reset_index(drop=True)

    # ==========================
    # Rename kolom PR
    # ==========================

    kolom_baru = []

    nama = None

    for c in df.columns:

        if c.endswith("_LK"):
            nama = c[:-3]
            kolom_baru.append(c)

        elif c == "PR":
            kolom_baru.append(f"{nama}_PR")

        else:
            kolom_baru.append(c)

    df.columns = (
        pd.Index(kolom_baru)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # ==========================
    # Kecamatan saja
    # ==========================

    df = df[df.iloc[:,0].isin(["KECAMATAN"])].reset_index(drop=True)

    # numerik
    for c in df.columns[2:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # ==========================
    # Seluruh pekerjaan perempuan
    # ==========================

    tidak_bekerja = [
        "BELUM_TIDAK_BEKERJA_PR",
        "MENGURUS_RUMAH_TANGGA_PR",
        "PELAJAR_MAHASISWA_PR",
        "PENSIUNAN_PR"
    ]

    bekerja_cols = [
        c
        for c in df.columns
        if c.endswith("_PR")
        and c not in tidak_bekerja
    ]

    df["Perempuan_Bekerja"] = df[bekerja_cols].sum(axis=1)

    temp = df[[
        "NAMA_KELURAHAN",
        "Perempuan_Bekerja"
    ]].copy()

    temp.columns = [
        "Kecamatan",
        "Perempuan_Bekerja"
    ]

    temp["Tahun"] = tahun

    gabungan.append(temp)

x4_final = pd.concat(gabungan, ignore_index=True)


C:\Users\Difta\AppData\Local\Temp\ipykernel_30400\872655877.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Perempuan_Bekerja"] = df[bekerja_cols].sum(axis=1)
C:\Users\Difta\AppData\Local\Temp\ipykernel_30400\872655877.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Perempuan_Bekerja"] = df[bekerja_cols].sum(axis=1)
C:\Users\Difta\AppData\Local\Temp\ipykernel_30400\872655877.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [365]:
jumlah_pdd_2023 = pd.read_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja\Jumlah_Penduduk_Surabaya_Jenis_Kelamin_2023.csv")
jumlah_pdd_2024 = pd.read_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja\Jumlah_Penduduk_Surabaya_Jenis_Kelamin_2024.csv")
jumlah_pdd_2025 = pd.read_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja\Jumlah_Penduduk_Surabaya_Jenis_Kelamin_2025.csv")

In [366]:
jumlah_pdd_2023["Tahun"] = 2023
jumlah_pdd_2024["Tahun"] = 2024
jumlah_pdd_2025["Tahun"] = 2025

In [367]:
jumlah_pdd_2023_2025 = pd.concat([jumlah_pdd_2023, jumlah_pdd_2024, jumlah_pdd_2025], ignore_index=True)

In [368]:
jumlah_perempuan = jumlah_pdd_2023_2025[["Kecamatan", "Tahun", "Perempuan"]].copy()

jumlah_perempuan = jumlah_perempuan.rename(columns={
    "Perempuan": "Jumlah Perempuan"
})

jumlah_perempuan["Kecamatan"] = (
    jumlah_perempuan["Kecamatan"]
    .str.upper()
    .str.strip()
)

jumlah_perempuan

,Kecamatan,Tahun,Jumlah Perempuan
0,GAYUNGAN,2023,22514
1,BULAK,2023,23669
2,ASEM ROWO,2023,23814
3,JAMBANGAN,2023,27373
4,GENTENG,2023,29963
...,...,...,...
88,WONOKROMO,2025,77231
89,SEMAMPIR,2025,90682
90,KENJERAN,2025,92631
91,SAWAHAN,2025,99622


In [369]:
x4_final = x4_final.merge(
    jumlah_perempuan,
    on=["Kecamatan","Tahun"],
    how="left"
)

x4_final["Persentase Perempuan Bekerja"] = (
    x4_final["Perempuan_Bekerja"]
    /
    x4_final["Jumlah Perempuan"]
    *100
)

x4_final.to_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Persentase perempuan yang bekerja\X4_Persentase Perempuan Bekerja",
                index=False,
                encoding="utf-8-sig")
x4_final.head()

,Kecamatan,Perempuan_Bekerja,Tahun,Jumlah Perempuan,Persentase Perempuan Bekerja
0,KARANG PILANG,2037,2023,38211,5.330926
1,WONOCOLO,2119,2023,40529,5.228355
2,RUNGKUT,3497,2023,61757,5.662516
3,WONOKROMO,5221,2023,78994,6.609363
4,TEGALSARI,3236,2023,49888,6.486530


# X5

In [407]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X5 Persentase Perempuan Lulus Wajib Belajar"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x5_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [420]:
gabungan = []

for tahun in range(2023, 2026):

    df = globals()[f"x5_{tahun}"].copy()

    temp = pd.DataFrame({
        "Kecamatan": df.iloc[:, 0],
        "Tahun": tahun,
        "Perempuan Lulus Wajib Belajar": (
            df.iloc[:, 10] +   # SLTA Perempuan
            df.iloc[:, 12] +   # D1/D2 Perempuan
            df.iloc[:, 14] +   # D3 Perempuan
            df.iloc[:, 16] +   # D4/S1 Perempuan
            df.iloc[:, 18] +   # S2 Perempuan
            df.iloc[:, 20]     # S3 Perempuan
        )
    })

    gabungan.append(temp)

# Gabungkan seluruh tahun
x5_final = pd.concat(gabungan, ignore_index=True)

# Normalisasi nama kecamatan
x5_final = normalisasi_kecamatan(x5_final)
jumlah_perempuan = normalisasi_kecamatan(jumlah_perempuan)

# Merge jumlah perempuan
x5_final = x5_final.merge(
    jumlah_perempuan,
    on=["Kecamatan", "Tahun"],
    how="left"
)

# Hitung persentase
x5_final["Persentase Perempuan Lulus Wajib Belajar"] = (
    x5_final["Perempuan Lulus Wajib Belajar"]
    / x5_final["Jumlah Perempuan"]
) * 100

# Susun kolom
x5_final = x5_final[[
    "Kecamatan",
    "Tahun",
    "Perempuan Lulus Wajib Belajar",
    "Jumlah Perempuan",
    "Persentase Perempuan Lulus Wajib Belajar"
]]

x5_final.to_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X5 Persentase Perempuan Lulus Wajib Belajar\X5_Persentase Perempuan Lulus.csv",
                index=False,
                encoding="utf-8-sig")

x5_final.head()

,Kecamatan,Tahun,Perempuan Lulus Wajib Belajar,Jumlah Perempuan,Persentase Perempuan Lulus Wajib Belajar
0,KARANG PILANG,2023,17669,38211,46.240611
1,JAMBANGAN,2023,13694,27373,50.027399
2,GAYUNGAN,2023,11586,22514,51.461313
3,WONOCOLO,2023,18462,40529,45.552567
4,TENGGILIS MEJOYO,2023,14201,29996,47.342979
